In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import pandas as pd
import numpy as np

In [3]:
import sqlalchemy as sa
from sqlalchemy.orm import Session
from crmprtd import setup_logging
from pycds import Station, History

save_path = './comparison_forms/'

db_url = "postgresql+psycopg2://tongli1997@/crmp?host=pg01.pcic.uvic.ca,pg02.pcic.uvic.ca&port=5432,5432&target_session_attrs=read-write&passfile=/workspaces/crmprtd/.pgpass"
log_file_path = save_path

engine = sa.create_engine(db_url, echo=False)
session = Session(engine)

session

In [4]:
path = '/workspaces/crmprtd/update_round2/input_tables/'
df = pd.read_excel(path + '20251223-Metadata-AllRequiredChanges_round2.xlsx', header = 1)   # pandas automatically uses openpyxl



### Solved the issue
    "Additional - Responsibility Change - move any data from 2436 collected from Octo0ber 1, 2022 to this site",
    'Site CLOSED - Move all data collected post October 1, 2022 to new ENV-ASP site "Breanda Mines" Line 1718'

ENV-ASP is BC-Snow


In [5]:
wanted_issues = [
    "Additional - Responsibility Change - move any data from 2436 collected from Octo0ber 1, 2022 to this site",
    'Site CLOSED - Move all data collected post October 1, 2022 to new ENV-ASP site "Breanda Mines" Line 1718'
]

df_2 = df[
    df["ISSUE"].str.strip().isin(wanted_issues) 
]

df_2 = df_2[['ISSUE', 'Station ID', 'Network ID', 'Network Name', 'Native ID', 'Unique Names', 'Unique Location (String)']].reset_index(drop=True)

pattern = r'->\s*([0-9\.\-]+)\s*W,\s*([0-9\.\-]+)\s*N,\s*Elev\.\s*([0-9\.\-]+)\s*m'

df_2[['lon', 'lat', 'elev']] = df_2['Unique Location (String)'].str.extract(pattern).astype(float)

df_2["Native ID"] = df_2["Native ID"].str.replace(r'.*->\s*', '', regex=True)
df_2 = df_2.drop(columns=['Unique Location (String)'])

df_2.iloc[1]

ISSUE           Site CLOSED - Move all data collected post Oct...
Station ID                                                 2436.0
Network ID                                                    5.0
Network Name                               BCH-GSO-HMP -> ENV ASP
Native ID                                                     BMN
Unique Names                                         Brenda Mines
lon                                                           NaN
lat                                                           NaN
elev                                                          NaN
Name: 1, dtype: object

In [ ]:
from sqlalchemy import func

stations_created = []

# Get the first row as a Series
row = df_2.iloc[0]

name = row['Unique Names']
nid  = row['Native ID']

# 1. Create Station
st = Station(
    native_id=nid,
    publish=True,
    network_id=14
)

session.add(st)
session.flush()  # get st.id

h = History(
    station_id=st.id,
    station_name=name,
    lat=float(row['lat']),
    lon=float(row['lon']),
    elevation=float(row['elev']),
    province="BC",
    country="CA",
    the_geom=func.ST_SetSRID(func.ST_MakePoint(float(row['lon']), float(row['lat'])), 4326)
)

session.add(h)
session.commit()
stations_created.append((name, st.id))
print("Created:", stations_created)

Created: [('Brenda Mines', 14960)]


In [8]:
df_2

,ISSUE,Station ID,Network ID,Network Name,Native ID,Unique Names,lon,lat,elev
0,Additional - Responsibility Change - move any ...,NaN,NaN,BC-Snow,2F18P,Brenda Mines,119.988889,49.868056,1460.0
1,Site CLOSED - Move all data collected post Oct...,2436.0,5.0,BCH-GSO-HMP -> ENV ASP,BMN,Brenda Mines,NaN,NaN,NaN


In [9]:
from sqlalchemy import text

# 1️⃣ Query to select old observations that need moving
SQL_TO_MOVE = text("""
SELECT o_old.history_id AS old_hist_id,
       o_old.obs_time,
       o_old.vars_id
FROM obs_raw o_old
JOIN meta_history h_old ON o_old.history_id = h_old.history_id
JOIN meta_station s_old ON h_old.station_id = s_old.station_id
WHERE s_old.native_id = :old_id

EXCEPT

SELECT o_new.history_id AS old_hist_id,
       o_new.obs_time,
       o_new.vars_id
FROM obs_raw o_new
JOIN meta_history h_new ON o_new.history_id = h_new.history_id
JOIN meta_station s_new ON h_new.station_id = s_new.station_id
WHERE s_new.native_id = :new_id
""")

# 2️⃣ Get the new station's history_id (assume we use latest)
SQL_NEW_HISTORY = text("""
SELECT h.history_id
FROM meta_history h
JOIN meta_station s ON h.station_id = s.station_id
WHERE s.native_id = :new_id
  AND s.station_id = (
      SELECT MAX(station_id)
      FROM meta_station
      WHERE native_id = :new_id
  )
ORDER BY h.history_id DESC
LIMIT 1
""")

def move_station_obs_fast(old_id, new_id, engine):
    with engine.begin() as conn:
        # get new history_id (latest)
        new_hist_id = conn.execute(SQL_NEW_HISTORY, {"new_id": new_id}).scalar()
        if new_hist_id is None:
            print(f"New station '{new_id}' has no history, skipping.")
            return 0

        # update all in one query
        n_move = conn.execute(
            text("""
                WITH updated AS (
                    UPDATE obs_raw o
                    SET history_id = :new_hist_id
                    FROM meta_history h_old
                    JOIN meta_station s_old ON h_old.station_id = s_old.station_id
                    WHERE o.history_id = h_old.history_id
                      AND s_old.native_id = :old_id
                      AND o.obs_time >= '2022-10-01'  -- only move obs after this date
                      AND NOT EXISTS (
                          SELECT 1
                          FROM obs_raw o_new
                          JOIN meta_history h_new ON o_new.history_id = h_new.history_id
                          JOIN meta_station s_new ON h_new.station_id = s_new.station_id
                          WHERE s_new.native_id = :new_id
                            AND o_new.obs_time = o.obs_time
                            AND o_new.vars_id = o.vars_id
                      )
                    RETURNING o.*
                )
                SELECT COUNT(*) FROM updated;
            """),
            {"old_id": old_id, "new_id": new_id, "new_hist_id": new_hist_id}
        ).scalar()

        print(f"Moved {n_move} rows from '{old_id}' to '{new_id}'")
        return n_move


# 4️⃣ Loop through your dataframe
results = []

old_id = df_2['Native ID'].iloc[1]
new_id = df_2['Native ID'].iloc[0]
print(f" Processing old='{old_id}' -> new='{new_id}'")
n = move_station_obs_fast(old_id, new_id, engine)
results.append(n)

# df_concat_new['n_moved'] = results
print("All done!")

 Processing old='BMN' -> new='2F18P'
Moved 71327 rows from 'BMN' to '2F18P'
All done!


### mapping the variable to the ones in the new network, then update the network id in the obs_raw table

!!!! Manually change the station_id here

In [6]:
from sqlalchemy import text
import pandas as pd

SQL_VARS_INFO = text("""
SELECT DISTINCT
       -- o.vars_id,
       v.*,
       -- m.station_id,
       s.native_id
       -- m.station_name,
       -- s.network_id AS station_network_id
FROM meta_history AS m
JOIN meta_station AS s ON m.station_id = s.station_id
JOIN obs_raw AS o ON m.history_id = o.history_id
JOIN meta_vars AS v ON o.vars_id = v.vars_id
WHERE s.station_id = :station_id
""")

df_vars = pd.read_sql(
    SQL_VARS_INFO,
    engine,
    params={"station_id": 14960}
)

df_vars

,vars_id,network_id,unit,precision,standard_name,cell_method,long_description,net_var_name,display_name,short_name,mod_time,mod_user,native_id
0,465,5,mm,None,lwe_thickness_of_precipitation_amount,time: sum,Daily precipitation,ONE_DAY_PRECIPITATION,Precipitation Amount,lwe_thickness_of_precipitation_amount_sum,2025-02-11 16:03:39.747374,crmp,2F18P
1,468,5,cm,None,surface_snow_thickness,time: point,Amount of snow on the ground,SNOW_ON_THE_GROUND,Surface Snow Depth (Point),surface_snow_thickness_point,2025-02-11 16:03:39.747374,crmp,2F18P
2,657,5,celsius,None,air_temperature,time: point,Hourly air temperature instantaneous,AirTemp,Temperature (Point),air_temperature_point,2025-02-11 16:03:39.747374,crmp,2F18P
3,463,5,celsius,None,air_temperature,time: minimum,Minimum daily temperature,MIN_TEMP,Temperature (Min.),air_temperature_minimum,2025-02-11 16:03:39.747374,crmp,2F18P
4,464,5,celsius,None,air_temperature,time: maximum,Maximum daily temperature,MAX_TEMP,Temperature (Max.),air_temperature_maximum,2025-02-11 16:03:39.747374,crmp,2F18P
5,659,5,mm,None,liquid_water_content_of_surface_snow,time: point,Snow water equivalent of snow on the ground,Snow_WE,Snow Water Equivalent,swe,2025-02-11 16:03:39.747374,crmp,2F18P


In [7]:
SQL_VARS_NET14 = text("""
SELECT DISTINCT
       v.vars_id,
       v.unit,
       v.precision,
       v.standard_name,
       v.cell_method,
       v.long_description,
       v.net_var_name,
       v.display_name,
       v.short_name
FROM meta_vars v
WHERE v.network_id = :network_id
""")

df_vars_net14 = pd.read_sql(
    SQL_VARS_NET14,
    engine,
    params={"network_id": 14}
)

In [8]:
df_vars_net14

,vars_id,unit,precision,standard_name,cell_method,long_description,net_var_name,display_name,short_name
0,513,mm,None,lwe_thickness_of_precipitation_amount,time: sum,Daily precipitation,ONE_DAY_PRECIPITATION,Precipitation Amount,lwe_thickness_of_precipitation_amount_sum
1,705,mm,None,lwe_thickness_of_precipitation_amount,time: sum,None,cum_pcpn_amt,Cumulative Precipitation,ppt_tot
2,710,mm,None,lwe_thickness_of_precipitation_amount,time: sum (interval: 24 hour),None,pcpn_amt_pst24hrs,Precipitation 24hr,ppt_24hr
3,702,celsius,None,air_temperature,time: point,None,air_temp_2,Temperature (Mean),T
4,660,mm,None,liquid_water_content_of_surface_snow,time: point,Snow water equivalent of snow on the ground,SWE,Snow Water Equivalent,swe
5,514,mm,None,thickness_of_rainfall_amount,time: sum,Daily rainfall,ONE_DAY_RAIN,Rainfall Amount,thickness_of_rainfall_amount_sum
6,515,cm,None,thickness_of_snowfall_amount,time: sum,Daily snow accumulation,ONE_DAY_SNOW,Snowfall Amount,thickness_of_snowfall_amount_sum
7,706,mm,None,lwe_thickness_of_precipitation_amount,time: sum (interval: 1 hour),None,pcpn_amt_pst1hr,Precipitation 1hr,ppt
8,600,mm,None,lwe_thickness_of_precipitation_amount,t: sum within months t: mean over years,Climatological mean of monthly total precipita...,Precip_Climatology,Precipitation Climatology,lwe_thickness_of_precipitation_amountt: sum wi...
9,512,celsius,None,air_temperature,time: maximum,Maximum daily temperature,MAX_TEMP,Temperature (Max.),air_temperature_maximum


In [9]:
VAR_MATCH_COLS = [
    "unit",
    "precision",
    "standard_name",
    "cell_method",
    "long_description",
    "net_var_name",
    "display_name",
    "short_name",
]

# Merge again, but keep vars_id from network 14
df_compare = df_vars.merge(
    df_vars_net14[VAR_MATCH_COLS + ["vars_id"]],  # only keep vars_id from net14
    on=VAR_MATCH_COLS,
    how="left",
    suffixes=("", "_net14"),
    indicator=True
)

# Add a boolean column to mark existence
df_compare["exists_in_net14"] = df_compare["_merge"] == "both"

# Rename vars_id from network 14 for clarity
df_compare["vars_id_net14"] = df_compare["vars_id_net14"]

df_compare

,vars_id,network_id,unit,precision,standard_name,cell_method,long_description,net_var_name,display_name,short_name,mod_time,mod_user,native_id,vars_id_net14,_merge,exists_in_net14
0,465,5,mm,None,lwe_thickness_of_precipitation_amount,time: sum,Daily precipitation,ONE_DAY_PRECIPITATION,Precipitation Amount,lwe_thickness_of_precipitation_amount_sum,2025-02-11 16:03:39.747374,crmp,2F18P,513.0,both,True
1,468,5,cm,None,surface_snow_thickness,time: point,Amount of snow on the ground,SNOW_ON_THE_GROUND,Surface Snow Depth (Point),surface_snow_thickness_point,2025-02-11 16:03:39.747374,crmp,2F18P,516.0,both,True
2,657,5,celsius,None,air_temperature,time: point,Hourly air temperature instantaneous,AirTemp,Temperature (Point),air_temperature_point,2025-02-11 16:03:39.747374,crmp,2F18P,NaN,left_only,False
3,463,5,celsius,None,air_temperature,time: minimum,Minimum daily temperature,MIN_TEMP,Temperature (Min.),air_temperature_minimum,2025-02-11 16:03:39.747374,crmp,2F18P,511.0,both,True
4,464,5,celsius,None,air_temperature,time: maximum,Maximum daily temperature,MAX_TEMP,Temperature (Max.),air_temperature_maximum,2025-02-11 16:03:39.747374,crmp,2F18P,512.0,both,True
5,659,5,mm,None,liquid_water_content_of_surface_snow,time: point,Snow water equivalent of snow on the ground,Snow_WE,Snow Water Equivalent,swe,2025-02-11 16:03:39.747374,crmp,2F18P,NaN,left_only,False


In [59]:
658. 657

658,celsius,,air_temperature,time: point,Hourly air temperature instantaneous,Temp,Temperature (Point),air_temperature_point
657,celsius,,air_temperature,time: point,Hourly air temperature instantaneous,AirTemp,Temperature (Point),air_temperature_point


660. 659

660,mm,,liquid_water_content_of_surface_snow,time: point,Snow water equivalent of snow on the ground,SWE,Snow Water Equivalent,swe
659,mm,,liquid_water_content_of_surface_snow,time: point,Snow water equivalent of snow on the ground,Snow_WE,Snow Water Equivalent,swe


SyntaxError: invalid syntax (3274095661.py, line 1)

In [10]:
df_compare.loc[df_compare["vars_id"] == 657, "vars_id_net14"] = 658
df_compare.loc[df_compare["vars_id"] == 659, "vars_id_net14"] = 660

df_compare

,vars_id,network_id,unit,precision,standard_name,cell_method,long_description,net_var_name,display_name,short_name,mod_time,mod_user,native_id,vars_id_net14,_merge,exists_in_net14
0,465,5,mm,None,lwe_thickness_of_precipitation_amount,time: sum,Daily precipitation,ONE_DAY_PRECIPITATION,Precipitation Amount,lwe_thickness_of_precipitation_amount_sum,2025-02-11 16:03:39.747374,crmp,2F18P,513.0,both,True
1,468,5,cm,None,surface_snow_thickness,time: point,Amount of snow on the ground,SNOW_ON_THE_GROUND,Surface Snow Depth (Point),surface_snow_thickness_point,2025-02-11 16:03:39.747374,crmp,2F18P,516.0,both,True
2,657,5,celsius,None,air_temperature,time: point,Hourly air temperature instantaneous,AirTemp,Temperature (Point),air_temperature_point,2025-02-11 16:03:39.747374,crmp,2F18P,658.0,left_only,False
3,463,5,celsius,None,air_temperature,time: minimum,Minimum daily temperature,MIN_TEMP,Temperature (Min.),air_temperature_minimum,2025-02-11 16:03:39.747374,crmp,2F18P,511.0,both,True
4,464,5,celsius,None,air_temperature,time: maximum,Maximum daily temperature,MAX_TEMP,Temperature (Max.),air_temperature_maximum,2025-02-11 16:03:39.747374,crmp,2F18P,512.0,both,True
5,659,5,mm,None,liquid_water_content_of_surface_snow,time: point,Snow water equivalent of snow on the ground,Snow_WE,Snow Water Equivalent,swe,2025-02-11 16:03:39.747374,crmp,2F18P,660.0,left_only,False


### update the variable id for the new station

In [11]:
df_compare

,vars_id,network_id,unit,precision,standard_name,cell_method,long_description,net_var_name,display_name,short_name,mod_time,mod_user,native_id,vars_id_net14,_merge,exists_in_net14
0,465,5,mm,None,lwe_thickness_of_precipitation_amount,time: sum,Daily precipitation,ONE_DAY_PRECIPITATION,Precipitation Amount,lwe_thickness_of_precipitation_amount_sum,2025-02-11 16:03:39.747374,crmp,2F18P,513.0,both,True
1,468,5,cm,None,surface_snow_thickness,time: point,Amount of snow on the ground,SNOW_ON_THE_GROUND,Surface Snow Depth (Point),surface_snow_thickness_point,2025-02-11 16:03:39.747374,crmp,2F18P,516.0,both,True
2,657,5,celsius,None,air_temperature,time: point,Hourly air temperature instantaneous,AirTemp,Temperature (Point),air_temperature_point,2025-02-11 16:03:39.747374,crmp,2F18P,658.0,left_only,False
3,463,5,celsius,None,air_temperature,time: minimum,Minimum daily temperature,MIN_TEMP,Temperature (Min.),air_temperature_minimum,2025-02-11 16:03:39.747374,crmp,2F18P,511.0,both,True
4,464,5,celsius,None,air_temperature,time: maximum,Maximum daily temperature,MAX_TEMP,Temperature (Max.),air_temperature_maximum,2025-02-11 16:03:39.747374,crmp,2F18P,512.0,both,True
5,659,5,mm,None,liquid_water_content_of_surface_snow,time: point,Snow water equivalent of snow on the ground,Snow_WE,Snow Water Equivalent,swe,2025-02-11 16:03:39.747374,crmp,2F18P,660.0,left_only,False


## !!!! Manually change the station_id here


In [13]:

# 2️⃣ Build a list of (old_vars_id, new_vars_id) tuples
mapping_list = list(df_compare[["vars_id", "vars_id_net14"]].itertuples(index=False, name=None))
mapping_list

new_station_id = 14960


SQL_UPDATE_VARS = text("""
UPDATE obs_raw o
SET vars_id = :new_vars_id
FROM meta_history h
JOIN meta_station s ON h.station_id = s.station_id
WHERE o.history_id = h.history_id
  AND s.station_id = :station_id
  AND o.vars_id = :old_vars_id
""")

with engine.begin() as conn:
    for old_var, new_var in mapping_list:
        res = conn.execute(
            SQL_UPDATE_VARS,
            {
                "station_id": new_station_id,  # your new station id
                "old_vars_id": old_var,
                "new_vars_id": new_var,
            }
        )
        print(f"Updated vars_id {old_var} → {new_var}, rows affected: {res.rowcount}")



Updated vars_id 465 → 513.0, rows affected: 1040
Updated vars_id 468 → 516.0, rows affected: 20173
Updated vars_id 657 → 658.0, rows affected: 28197
Updated vars_id 463 → 511.0, rows affected: 1040
Updated vars_id 464 → 512.0, rows affected: 1040
Updated vars_id 659 → 660.0, rows affected: 19837
